<a href="https://colab.research.google.com/github/takuonakashima/ai-security-workshop/blob/main/cifar_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. データの前処理と読み込み (CIFAR-10用に変更)
# ==========================================
# カラー画像(RGB)なので、Normalizeのパラメータが3チャンネル分(0.5, 0.5, 0.5)になります
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# MNIST -> CIFAR10 に変更
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# CIFAR-10の10個のクラス名（正解ラベル）
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# ==========================================
# 2. モデルの定義 (CIFAR-10 カラー画像対応版)
# ==========================================
class SimpleCNN_CIFAR(nn.Module):
    def __init__(self):
        super(SimpleCNN_CIFAR, self).__init__()
        # 変更点1：入力チャンネルを1(白黒) -> 3(RGBカラー)に変更
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()

        # 変更点2：画像サイズが32x32。プーリングを2回通ると 32 -> 16 -> 8 になる。
        # 64チャンネル × 8 × 8 = 4096
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10) # 最終的な10クラス分類

    def forward(self, x):
        x = self.maxpool(self.relu(self.conv1(x)))
        x = self.maxpool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN_CIFAR()

# ==========================================
# 3. 学習の準備
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"使用デバイス: {device}")

# ==========================================
# 4. 学習ループ (演習用に3エポックで短く設定)
# ==========================================
epochs = 3
print("\n--- CIFAR-10 学習を開始します ---")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# ==========================================
# 5. モデルの評価と可視化
# ==========================================
print("\n--- 学習完了。推論テストを行います ---")
model.eval()

# テストデータから1バッチ取得
dataiter = iter(test_loader)
images, labels = next(dataiter)
images_cpu, labels_cpu = images.to(device), labels.to(device)

# 推論の実行
with torch.no_grad():
    outputs = model(images_cpu)
    _, predicted = torch.max(outputs, 1)

# 結果の可視化関数（正規化を解除してカラー画像として表示）
def imshow(img):
    img = img / 2 + 0.5 # [-1, 1] を [0, 1] に戻す
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

# 最初の5枚を表示して確認
fig = plt.figure(figsize=(12, 4))
for i in range(5):
    ax = fig.add_subplot(1, 5, i+1, xticks=[], yticks=[])
    imshow(images[i])
    # 予測結果と実際のラベル（クラス名）を表示
    pred_name = classes[predicted[i].item()]
    true_name = classes[labels[i].item()]

    color = "green" if pred_name == true_name else "red"
    ax.set_title(f"Pred: {pred_name}\nTrue: {true_name}", color=color)

plt.tight_layout()
plt.show()